In [ ]:
pip install tensorflow

In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

import numpy
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.model_selection import GroupKFold
from scipy import stats

In [ ]:
Water_Quality_df = pd.read_csv("data/water_quality_training_dataset.csv")
landsat_train_features = pd.read_csv("data/landsat_features_training_combined.csv")
Terraclimate_df = pd.read_csv("data/terraclimate_features_training_combined.csv")

#Clean Up the Data
#Capitalize Col Names
def capitalize_words_keep_spacing(col):
    # Split on space or underscore but keep the separators
    parts = re.split(r'([ _])', col)
    # Capitalize text parts, keep separators unchanged
    return ''.join(p.title() if p not in [' ', '_'] else p for p in parts)
Terraclimate_df.columns = [capitalize_words_keep_spacing(c) for c in Terraclimate_df.columns]
landsat_train_features.columns = [capitalize_words_keep_spacing(c) for c in landsat_train_features.columns]
Water_Quality_df.columns = [capitalize_words_keep_spacing(c) for c in Water_Quality_df.columns]

def combine_two_datasets(dataset1,dataset2,dataset3):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([dataset1,dataset2,dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)

#Data type corrections 
wq_data['Sample Date'] = pd.to_datetime(wq_data['Sample Date'],  format='%d-%m-%Y')


def convert_text_cols_to_float(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if (pd.api.types.is_object_dtype(out[c]) 
            or pd.api.types.is_string_dtype(out[c]) 
            or pd.api.types.is_categorical_dtype(out[c])):
            s = out[c].astype(str).str.replace('\u00A0', ' ', regex=False).str.strip()
            s = s.str.replace('\u2212', '-', regex=False)                 # unicode minus
            s = s.str.replace(r'^\(\s*(.*)\s*\)$', r'-\1', regex=True)    # (123) -> -123
            s = s.str.replace(r'^\s*([+]?\s*[\d, .]+)\s*-$', r'-\1', regex=True)  # 123- -> -123
            s = s.str.replace(r'(?<=\d),(?=\d{3}\b)', '', regex=True)      # thousands comma
            out[c] = pd.to_numeric(s, errors='coerce')                     # float64 by default with NaNs
    return out

wq_data = convert_text_cols_to_float(wq_data)

#ullify all negative observations
for column in wq_data.columns:
    if column != "Sample Date": wq_data[wq_data[column] < -9000][column] = np.nan 

wq_data['Month_cosine'] = np.cos((wq_data['Sample Date'].dt.month + (wq_data['Sample Date'].dt.day/31))* np.pi / 6)
plt.scatter(wq_data['Sample Date'], wq_data['Month_cosine'])
plt.xlabel('Sample Date')
plt.ylabel('Month_Cosine')
plt.show()

wq_data = wq_data.drop(columns=['Qa_Radsat', 'Cloud_Qa', 'Sample Date'])
wq_data = wq_data.dropna(how='any',axis=0)

#number of cv groups
cv_groups = 8

#split over longitude
wq_data['cv_group'] = pd.qcut(wq_data['Latitude'], q=cv_groups, labels=False)

print(wq_data.info())

plt.scatter(wq_data['Latitude'], wq_data['cv_group'])
plt.xlabel('Longitude')
plt.ylabel('cv_group')
plt.show()

# Specify the number of folds
lat_sep_kf = GroupKFold(n_splits=cv_groups - 1)

In [ ]:
#Box Cox of predictors
Total_Alkalinity_bc, Total_Alkalinity_lambda_opt = stats.boxcox(wq_data['Total Alkalinity'])
Electrical_Conductance_bc, Electrical_Conductance_lambda_opt = stats.boxcox(wq_data['Electrical Conductance'])
Dissolved_Reactive_Phosphorus_bc, Dissolved_Reactive_Phosphorus_lambda_opt = stats.boxcox(wq_data['Dissolved Reactive Phosphorus'])

wq_data['Total Alkalinity'] = Total_Alkalinity_bc
wq_data['Electrical Conductance'] = Electrical_Conductance_bc
wq_data['Dissolved Reactive Phosphorus'] = Dissolved_Reactive_Phosphorus_bc

#Square x value
squarelwir1 = wq_data['Lwir'] ** 2
wq_data['lwir'] = squarelwir1

In [ ]:
#test train based on location
wq_data_test = wq_data[wq_data['cv_group'] == 4]
wq_data_train = wq_data[wq_data['cv_group'] != 4]

#then split into X and Y 
Y_train = wq_data_train[["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]]
X_train = wq_data_train.drop(columns=["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus", "Longitude", "Latitude"])
Y_test = wq_data_test[["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]]
X_test = wq_data_test.drop(columns=["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus", "Longitude", "Latitude"])

'''
Y_data = wq_data[["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]]
X_data = wq_data.drop(columns=["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus", "Longitude", "Latitude"])

X_train, X_test, Y_train, Y_test = train_test_split(
    X_data, Y_data, test_size=0.2, random_state=88, shuffle=False
)
'''

print(Y_train.shape)
print(X_train.shape)
print(Y_test.shape)
print(X_test.shape)


In [ ]:
X_train = X_train[sorted(X_train.columns)]
X_test = X_test[sorted(X_test.columns)]

In [ ]:
X_train = X_train.drop('cv_group', axis= 1)
X_train = X_train.drop('lwir', axis= 1)
X_train = X_train.drop('Ndwi', axis= 1)

X_test = X_test.drop('cv_group', axis= 1)
X_test = X_test.drop('lwir', axis= 1)
X_test = X_test.drop('Ndwi', axis= 1)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model = keras.Sequential([
    layers.Dense(120, activation='relu', input_shape=(55,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(3, activation='relu')
])

In [ ]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [ ]:
history = model.fit(X_train, Y_train, validation_split=0.2, epochs=3, batch_size=55)

In [ ]:
loss, mae = model.evaluate(X_test, Y_test)
print("Test MAE:", mae)

In [ ]:
Water_Quality_df_v = pd.read_csv("data/submission_template.csv")
landsat_train_features_v = pd.read_csv("data/landsat_features_validation_combined.csv")
Terraclimate_df_v = pd.read_csv("data/terraclimate_features_validation_combined.csv")

#Clean Up the Data
Terraclimate_df_v.columns = [capitalize_words_keep_spacing(c) for c in Terraclimate_df_v.columns]
landsat_train_features_v.columns = [capitalize_words_keep_spacing(c) for c in landsat_train_features_v.columns]
Water_Quality_df_v.columns = [capitalize_words_keep_spacing(c) for c in Water_Quality_df_v.columns]

wq_data_v = combine_two_datasets(Water_Quality_df_v, landsat_train_features_v, Terraclimate_df_v)

wq_data_v.shape



In [ ]:
#Data type corrections 
wq_data_v['Sample Date'] = pd.to_datetime(wq_data_v['Sample Date'],  format='%d-%m-%Y')
wq_data_v = convert_text_cols_to_float(wq_data_v)

wq_data_v = wq_data_v.drop(columns=['Qa_Radsat', 'Cloud_Qa'])
#ullify all negative observations
for column in wq_data_v.columns:
    if column != "Sample Date": wq_data_v[wq_data_v[column] < -9000][column] = np.nan 

wq_data_v['Month_cosine'] = np.cos((wq_data_v['Sample Date'].dt.month + (wq_data_v['Sample Date'].dt.day/31))* np.pi / 6)
plt.scatter(wq_data_v['Sample Date'], wq_data_v['Month_cosine'])
plt.xlabel('Sample Date')
plt.ylabel('Month_Cosine')
plt.show()

#wq_data = wq_data.drop(columns=['Qa_Radsat', 'Cloud_Qa'], axis=1)
#wq_data_v = wq_data_v.dropna(how='any',axis=0)
#wq_data_v = wq_data_v.fillna(wq_data_v.median(numeric_only=True))

print(wq_data_v.info())

In [ ]:
wq_data_v_base = pd.DataFrame({
    'Latitude': wq_data_v['Latitude'].values,
    'Longitude': wq_data_v['Longitude'].values,
    'Sample Date': wq_data_v['Sample Date'].values
    })

In [ ]:

# type(wq_data_v)
wq_data_v = wq_data_v.drop('Latitude', axis= 1)
wq_data_v = wq_data_v.drop('Longitude', axis= 1)
wq_data_v = wq_data_v.drop('Sample Date', axis= 1)
# wq_data_v = wq_data_v.drop('Month_cosine', axis= 1)
wq_data_v = wq_data_v.drop('Total Alkalinity', axis= 1)
wq_data_v = wq_data_v.drop('Electrical Conductance', axis= 1)
wq_data_v = wq_data_v.drop('Dissolved Reactive Phosphorus', axis= 1)

In [ ]:
wq_data_v = wq_data_v[sorted(wq_data_v.columns)]

In [ ]:
wq_data_v = scaler.transform(wq_data_v)

In [ ]:
predictions = model.predict(wq_data_v)
display(predictions)


In [ ]:
df = pd.DataFrame(
    predictions,
    columns=["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]
    
    ,  # Optional column names
    )

In [ ]:
print(wq_data_v_base)

In [ ]:
submission_df = pd.DataFrame({
    'Longitude': wq_data_v_base["Latitude"].values,
    'Latitude': wq_data_v_base["Longitude"].values,
    'Sample Date': wq_data_v_base["Sample Date"].values,
    'Total Alkalinity': df["Total Alkalinity"].values,
    'Electrical Conductance': df["Electrical Conductance"].values,
    'Dissolved Reactive Phosphorus': df["Dissolved Reactive Phosphorus"].values
})

In [ ]:
submission_df.head(5)